# ContractLens — fine-tune Legal-BERT on CUAD (Colab GPU)

**Runtime → Change runtime type → T4 GPU**, then run all cells (~30–40 min).

Trains `nlpaueb/legal-bert-base-uncased` on exactly the same segments, labels and official test split as the TF-IDF baseline, so the two are directly comparable. Download `legalbert.zip` at the end and unzip it into `artifacts/legalbert/`, then serve it with `CONTRACTLENS_MODEL=artifacts/legalbert`.

In [ ]:
!git clone https://github.com/arihxnt25/Legal-doc-analyser.git
%cd Legal-doc-analyser
!pip install -q -r requirements-transformer.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) TF-IDF baseline (~5 min on Colab CPU cores)
!python scripts/train.py

In [ ]:
# 2) Legal-BERT (downloads CUAD + model weights automatically)
!python scripts/train_legalbert.py --epochs 2 --batch-size 32

In [ ]:
import json, pandas as pd
base = json.load(open('artifacts/metrics.json'))['summary']
bert = json.load(open('artifacts/legalbert/metrics.json'))['summary']
pd.DataFrame({'TF-IDF + LR': base, 'Legal-BERT': bert}).loc[['macro_aupr','macro_p_at_80r','macro_f1','micro_f1','macro_doc_f1','doc_accuracy']]

In [ ]:
# per-category comparison
b = pd.DataFrame(json.load(open('artifacts/metrics.json'))['per_category']).T['aupr']
l = pd.DataFrame(json.load(open('artifacts/legalbert/metrics.json'))['per_category']).T['aupr']
cmp = pd.DataFrame({'tfidf': b, 'legalbert': l}).astype(float)
cmp['delta'] = cmp.legalbert - cmp.tfidf
cmp.sort_values('delta')

In [ ]:
!cd artifacts && zip -qr ../legalbert.zip legalbert
from google.colab import files
files.download('legalbert.zip')